In [ ]:
from multiprocessing import Pool
import subprocess
import yaml
import os
import sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import tempfile
import pandas as pd
from collections import Counter
import numpy as np
import threading
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

from sklearn.metrics import accuracy_score
from joblib import load
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import cv2

In [ ]:
selected_FE = 'resnet50'  # Example feature extractor, can be changed
huggingface = False  # Set to True if using HuggingFace models
data_augmentation = False
suffix = '_augmented' if data_augmentation else ''
kind= 'patches_224'  # Example kind, can be changed
train_filename,_,_ = file_IO.load_input_files(source_path,selected_FE,kind,suffix)
mean,scale = model_utils.get_normalization_parameters(train_filename)

selected_classification_head = 'MLPClassifier1'
model_parameters = {
    'n_neurons':512,
    'dropout':0.2,
    'with_input_norm': 'batch_norm',  # Whether to use input normalization 'batch_norm' , None
    'mean': mean,  # Mean for input normalization
    'scale': scale,  # Scale for input normalization
}
head_type = 'pytorch'  # 'sklearn' or 'pytorch'
selected_metric = 'weighted_vote' #['majority_vote', 'weighted_vote', 'most_probable']
if kind == 'body':
    selected_metric = 'most_probable'  # Use the metric you want to analyze

#prepare folders to save results
base_dir=source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\representation_extraction'
if head_type == 'sklearn':
    search_dir = base_dir+f'\\sklearn_model_trained_on_rep\\{selected_classification_head}\\'
else:
    search_dir = base_dir+f'\\torch_model_trained_on_rep\\{selected_classification_head}\\'
search_dir+= 'images\\'
explanation_dir, cam_dir, attention_dir, original_dir, transformed_dir, augmented_dir = file_IO.init_expl_dirs(search_dir,clear=True)

is_train = 'test' #val, test, train
if is_train == 'val':
    zarr='test_public_writers'
elif is_train == 'test':
    zarr='test_private_writers'
else:
    zarr='train_writers'
zarr_path=f"C:\\Users\\andre\PhD\Datasets\ICDAR 2013 - Gender Identification Competition Dataset\\{zarr}.zarr"

In [ ]:
args = script_launching.DotDict(
    data_augmentation = data_augmentation,
    huggingface = huggingface,
    selected_FE = selected_FE,
    selected_classification_head= selected_classification_head,
    head_type = head_type,
    is_train = is_train,
    zarr_path = zarr_path,
    selected_metric = selected_metric,
    train_filename = train_filename,
    
)
file_IO.save_args(args,explanation_dir)  # Save the arguments to a file

In [ ]:
classification_head, val_df = data_loading.load_classification_head(selected_FE,selected_classification_head,
                                                                    head_type,'val_only',train_filename, source_path,**model_parameters)
transform = u_transforms.get_transform(selected_FE, use_patches=True, custom=False, mode='resize')
backbone = model_utils.get_model(name=selected_FE, mode='truncated', pretrained=True, truncation='remove head')
if head_type == 'sklearn':
    if selected_classification_head == 'logreg':
        # If the model is a logistic regression, wrap it in a PyTorch module
        classification_head = model_utils.SKLearnLogRegWrapper(classification_head.named_steps['logreg'])
    else:
        raise ValueError(f"Unsupported model type: {selected_classification_head}. Only 'logreg' is supported for now.")
model = model_utils.JoinedModels(backbone,classification_head)
selected=data_loading.load_selected_instances(source_path, selected_FE, selected_classification_head, head_type, selected_metric)

In [ ]:
zarr_visualizer=visualization.ZarrVisualizer(selected, zarr_path,selected_metric, search_dir, transform=transform, huggingface=huggingface, use_augmentation=True)
zarr_visualizer.save_images(mode='original')
zarr_visualizer.save_images(mode='preprocessed')
zarr_visualizer.save_images(mode='augmentation')

In [ ]:
#save explanations
mode='last'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cam_visualizations=explanation.gradcam_on_df(selected,model,transform,huggingface,device,selected_FE,mode=mode)

In [ ]:
visualization.display_vis_on_background(cam_visualizations,selected,selected_metric=selected_metric,blank_background=False,save_path=cam_dir)
#visualization.display_gradcam_vis(cam_visualizations,selected,selected_metric=selected_metric, save_path=cam_dir)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone = model_utils.get_model(name=selected_FE, mode='exp', pretrained=True, truncation='remove head')
attention_visualizations=explanation.attention_on_df(selected,backbone,transform,huggingface,device,model_name='clip-vit-large-patch14')

In [ ]:
visualization.display_vis_on_background(attention_visualizations,selected,selected_metric=selected_metric,blank_background=False,save_path=attention_dir)

# reload

In [ ]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    import utils.explanation as explanation
    import utils.script_launching as script_launching
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)
    importlib.reload(explanation)
    importlib.reload(script_launching)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, explanation, script_launching
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, explanation, script_launching = reload_modules()